**Notebook to Calibrate Wires trajectory for DAXM measurements from Fluorescence intensity shadowing**

- load a DAXM scan with visible shadows lines (scan performed on single sample point that produces fluorescence)
- GUI to find a set approached values for the N wires
- Refinement of Calibration parameters

**Section 2**: to start from scratch and select a daxm scan of interest

**Section 3**: GUI to find rather good parameters of the wires trajectory and positions. A .scan file is saved with these parameters in a created analysis directory. By default it is called `init_calib.scan`
This section can be skipped

**Section 4**: Calibration procedure, which can be started by loading `init_calib.scan` file.

#-------------------------------------------------------------

Initial version: Robin Freville based on Loic Renversade Code

Major Revision by Clement Ribart 2026

Last Revision by J.-S. Micha June 2026

# IMPORTS AND ENVIRONMENTS

In [ ]:
!which python

In [ ]:
import os, sys, ipympl
print(f'you are using pythn {sys.version}')
if sys.version_info.minor>=8:
    %matplotlib ipympl
    try:
        import ipympl
    except:
        print('You need to install ipympl: pip install ipympl')
else:
    %matplotlib notebook

In [ ]:
sys.executable, sys.version

In [ ]:
import os, time, copy
import itertools
import glob
from pathlib import Path

import matplotlib.pyplot as plt
#----------- global imports -----------

import matplotlib as mpl
import matplotlib.pylab as mplp
import matplotlib.pyplot as plt

import numpy as np

from __future__ import print_function
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets

In [ ]:
if 0: # test if we can get some plots (depending on notebook version ... and ipympl needs or not)
    fig_,ax_ = plt.subplots()
    ax_.plot(np.arange(20))

In [ ]:
# in jupyter-slurm for venv environment only (not conda !!  for this you need to create a ipykernel and)
# LaueToolsCode_Folder = '/data/bm32/inhouse/lauetoolsenv2/lib/python3.12/site-packages'
# sys.path.insert(0,LaueToolsCode_Folder)
# print(sys.path)
# import LaueTools as LT
# print('\ncode from', LT.__file__)

In [ ]:
import fabio
import pandas as pd
from pathlib import Path
from matplotlib.patches import Polygon
from matplotlib.lines import Line2D

In [ ]:
#----------- daxm imports -----------
from LaueTools.Daxm.classes.scan import new_scan
from LaueTools.Daxm.classes.source import new_source
#from LaueTools.Daxm.classes.calibration import CalibManager
#from LaueTools.Daxm.classes.reconstruction import SpotReconstructor
from LaueTools.Daxm.classes.scan.point import StaticPointScan, new_scan_dict, PointScan, save_scan_dict
import LaueTools.generaltools as GT
import LaueTools.logfile_reader as iohdf5
import LaueTools.blissdatafolderstructure as blissf
GT.__file__

In [ ]:
#print(mpl.get_backend())
#sys.version

In [ ]:
if 0: # test if we can get some plots (depending on notebook version ... and ipympl needs or not)
    fig_,ax_ = plt.subplots()
    ax_.plot(np.arange(20))

# SET DAXM SCAN, DETECTOR, USER FOLDER

In [ ]:
# example of the way data are organised in folders tree
# sample='Au15_1'
# dataset = '2Dmap_after'
# f'{sample}_{dataset}'
# f'/data/visitor/ma5820/bm32/20240208/RAW_DATA/{sample}/{sample}_{dataset}'

configurations = {
"blc17179": {
        "maindatafolder_RAW_DATA": "/data/visitor/blc17179/bm32/20260617/RAW_DATA/",
        "mainfolder_PROCESSED_DATA": "/data/visitor/blc17179/bm32/20260617/PROCESSED_DATA",
        "CCDType": "EIGER_4MCdTe",
        "Newsample_Name": "BaTiO3",
        "NewDataset_Name": "daxmfluo",
        "Scan_repo_Nb": "0001",
    },
"a321216": {
        "maindatafolder_RAW_DATA": "/data/visitor/a321216/bm32/20251008/RAW_DATA",
        "mainfolder_PROCESSED_DATA": "/data/visitor/a321216/bm32/20251008/PROCESSED_DATA",
        "CCDType": "sCMOS_4M",
        "Newsample_Name": "ech15_R3",
        "Zone": "Z1",
        "NewDataset_Name": "Z1_daxm_dia_combo_Line_1_10umstep",
        "Scan_repo_Nb": "0001",
    },

"blc16918": {
        "maindatafolder_RAW_DATA": "/data/visitor/blc16918/bm32/20260303/RAW_DATA",
        "mainfolder_PROCESSED_DATA": "/data/visitor/blc16918/bm32/20260303/PROCESSED_DATA",
        "CCDType": "EIGER_4MCdTe",
        "Newsample_Name": "ZrCr_ech15",
        "Zone": None,
        "NewDataset_Name": "aligndaxm",
        "Scan_repo_Nb": "0042",
    },
"blc16935": {
        "maindatafolder_RAW_DATA": "/data/visitor/blc16935/bm32/20260307/RAW_DATA",
        "mainfolder_PROCESSED_DATA": "/data/visitor/blc16935/bm32/20260307/PROCESSED_DATA",
        "CCDType": "EIGER_4MCdTe",
        "Newsample_Name": "ZrCr",
        "Zone": None,
        "NewDataset_Name": "Gedaxm",
        "Scan_repo_Nb": "0006",  # 501 pts
    },
    "utr20": {
        "maindatafolder_RAW_DATA": "/data/visitor/utr20/bm32/20260310/RAW_DATA/",
        "mainfolder_PROCESSED_DATA": "/data/visitor/utr20/bm32/20260310/PROCESSED_DATA",
        "CCDType": "EIGER_4MCdTe",
        "Newsample_Name": None,
        "Zone": None,
        "NewDataset_Name": None,
        "Scan_repo_Nb": None,  
    }
}


In [ ]:
# Select the configuration
expId = "blc17179"

# =================================================
config = configurations[expId]
GT.printgreen(f'selected config :\n {config}\n')
#Unpack the dictionary into variables
locals().update(config)
CCDLabel = CCDType

# Now you can use the variables directly:
print('Variables ----')
print('maindatafolder_RAW_DATA: ',maindatafolder_RAW_DATA)
print('CCDType: ',CCDType)
#print('NewDataset_Name: ',NewDataset_Name)

# check if images data folder exists:
imagefolder = Path(maindatafolder_RAW_DATA)/Newsample_Name/f'{Newsample_Name}_{NewDataset_Name}'/f'scan{Scan_repo_Nb}'
if not Path.exists(imagefolder):
    GT.printred(f'Folder {imagefolder} does not exist !!')
else:
    GT.printgreen(f'Folder {imagefolder} exists !!')

## Find automatically master .h5 file or SET local .h5 file or 

In [ ]:
mainh5file, HDF5_LOGFILE_EXISTS = blissf.findmasterh5file(maindatafolder_RAW_DATA)

print('mainh5file',mainh5file)

if mainh5file:
    GT.printgreen(f'Found master .h5 file')
    h5file = str(Path(f"{maindatafolder_RAW_DATA}")/mainh5file)
    GT.printgreen(f'set `h5file` to  : {h5file}')
else:
    GT.printyellow('H5 file not found! Check carefully the folder')
    HDF5_LOGFILE_EXISTS = False
    

In [ ]:
blissf.tree(maindatafolder_RAW_DATA,0)

In [ ]:
blissf.tree(Path(maindatafolder_RAW_DATA)/Newsample_Name,0, truncatesize=15)

In [ ]:
# [MAYBE NOT RECOMMENDED] sublevel sample h5 file
#h5file, HDF5_LOGFILE_EXISTS = blissf.findmasterh5file(Path(maindatafolder_RAW_DATA)/Newsample_Name)

# top level master h5 file
h5file, HDF5_LOGFILE_EXISTS = blissf.findmasterh5file(Path(maindatafolder_RAW_DATA))
h5file

## Read the hdf5 file

- get all scans given some constraints on corresponding dataset name
- get scans implying zf motors (daxm scans)

In [ ]:
HDF5_LOGFILE_EXISTS = True
# it can take a while if h5 file is large (many entries)
if HDF5_LOGFILE_EXISTS:
    logfile = iohdf5.H5file(h5file, CCDType)
    print('selected logfile .h5 :',logfile.path)
    logfile.getscans(verbose=1, n_last=10000, exclude_substrings=['grid'])
    dfallscans = logfile.dfallscans
    dfzfscans = logfile.filterby('motors', 'zf', openGUI=False)
    
    # print('pathHDF5',h5file)
    # listscans, listcts = iohdf5.get_scans_cts(h5file, verbose=0)
    # print("number scans found :",len(listscans))
    # print("number saved count commands found: ", len(listcts))
    # if len(listcts)>0:
    #     print("example of first count item : \n", listcts[0])

In [ ]:
dfzfscans

In [ ]:
#dfallscans

## SELECT scan of interest

set `hdf5_item_idx` to the scan of your choice in the first column

In [ ]:
# to see a specific item in the previous list (hdf5_item_idx on left column)
hdf5_item_idx = 463

d = logfile.build_dict_scan(hdf5_item_idx)
d
#==========================================================
#dfallscans.iloc[hdf5_item_idx], dfallscans['imagefolder'][hdf5_item_idx]

## SELECT a DAXM scan and SET parameters

it will create a StaticPointScan object that will be saved further in a ####_calib.scan

In [ ]:
# uncomment to be helped to browse over the good path
#!ls '/data/visitor/utr20/bm32/20260310/RAW_DATA/Al_mardisoir/Al_mardisoir_GeDAXM_heightstudy/scan0001/calibGe001_eiger4m.det'

In [ ]:
# select an index in the previous list (hdf5_item_idx on left column)
if expId == 'blc17179':
    
    number_wire = 3        # Number of wire used for DAXM
    calibdet_file = '/data/visitor/blc17179/bm32/20260617/RAW_DATA/Ge/Ge_0001/scan0002/calibGe001_Eiger_June2026_dist100mm.det'
    # calibdet_file = [80, 1079.6100, 983.0300, 0.2030000, 0.3220000, 0.07500000]  fake distance
    sizeofzeropadding = 4
    
    image_folder = d['imagefolder']
    prefix = d['prefix']
    suffix = '.'+d['suffix']
    samplename = str(d['samplename'])
    localhdf5file = d['localhdf5file']
    

if expId == 'utr20':
    prefix = "eiger4m_"
    suffix = '.h5'
    sizeofzeropadding = 4
    number_wire = 3         # Number of wire used for DAXM
    
    calibdet_file = '/data/visitor/utr20/bm32/20260310/RAW_DATA/Al_mardisoir/Al_mardisoir_GeDAXM_heightstudy/scan0001/calibGe001_eiger4m.det'
    # calibdet_file = [80, 1079.6100, 983.0300, 0.2030000, 0.3220000, 0.07500000]  fake distance
    image_folder = dfallscans['imagefolder'][hdf5_item_idx]


if expId == 'blc16935':
    prefix = "eiger4m_"
    suffix = '.h5'
    sizeofzeropadding = 4
    number_wire = 3         # Number of wire used for DAXM
    
    calibdet_file = '/data/visitor/blc16935/bm32/20260307/RAW_DATA/ZrCr/ZrCr_Gedaxm/scan0006/calibGe001_eiger4m_zcam100mm.det'
    # calibdet_file = [80, 1079.6100, 983.0300, 0.2030000, 0.3220000, 0.07500000]  fake distance
    image_folder = dfallscans['imagefolder'][hdf5_item_idx]

elif expId == 'blc16918':
    prefix = "eiger4m_"
    suffix = '.h5'
    sizeofzeropadding = 4
    number_wire = 4         # Number of wire used for DAXM
    calibdet_file = "/data/visitor/blc16918/bm32/20260303/RAW_DATA/ZrCr_ech15/ZrCr_ech15_Ge/scan0004/calibGe001_eiger4m_70mm_Ysmaller1500.det"
    image_folder = dfallscans['imagefolder'][hdf5_item_idx]



elif expId == 'a321216':
    calibdet_file = f"{maindatafolder_RAW_DATA}/Ge1/Ge1_calib_std/scan0001/Ge1.det"  # Calibration of detector geometry (with the Germanium)  .det file
    #image_folder = f'{maindatafolder_RAW_DATA}/ZrCr_ech15/ZrCr_ech15_aligndaxm/scan0042'     # Folder with images for the calibration of wire trajectory
    
    prefix = "img_" #prefix of RAW images  SCMOS
    suffix = '.tif'
    sizeofzeropadding = 4
    number_wire = 4         # Number of wire used for DAXM

#====================  rely on imagefolder=================================================================
raw_data_index = Path(image_folder).parts.index('RAW_DATA')
# The next two parts after 'RAW_DATA' are the first and second subfolders
Newsample_Name = Path(image_folder).parts[raw_data_index + 1]
NewDataset_Name = Path(image_folder).parts[raw_data_index + 2]
Scan_repo_Nb =  Path(image_folder).parts[raw_data_index + 3]
analysis_dir = f"{mainfolder_PROCESSED_DATA}/{Newsample_Name}/{NewDataset_Name}/{Scan_repo_Nb}"  ## Folder to write the _calib.scan file

localhdf5file = str(Path(blissf.extract_path_up_to_2_subfolders_after_raw_data(image_folder))/f'{NewDataset_Name}.h5')


# Make sure directory exists
os.makedirs(analysis_dir, exist_ok=True)
GT.printgreen(f'analysis_dir : {analysis_dir}')

scan_dict = new_scan_dict()

#====================  rely on dfallscans and ==hdf5_item_idx===========================================================

scantype = dfallscans['scantype'][hdf5_item_idx]
scanindex =  dfallscans['scanindex'][hdf5_item_idx]
sample_dataset_scanindex = dfallscans['sample_dataset_scanindex'][hdf5_item_idx]
fullcommand = dfallscans['fullcommand'][hdf5_item_idx]
#localhdf5file = dfallscans['localhdf5file'][hdf5_item_idx]  in case of manual entry of only imagefolder....

scan_number = scanindex #   scanindex    Scans start by 0001 - Look in column 'sample_dataset_scanindex' last digit = 'scanindex' column digit
ScanID = sample_dataset_scanindex #f"{Newsample_Name}_{NewDataset_Name}_{scan_number}"    

scan_dict["CCDType"] = CCDType 
scan_dict["specFile"] = h5file
scan_dict["scanNumber"] = scan_number
scan_dict['hdf5scanId'] = ScanID  #  hdf5 index
scan_dict["imageFolder"] = image_folder
scan_dict["imagePrefix"] = prefix
scan_dict["detCalib"] = calibdet_file
scan_dict['wire'] = [['W',0.025, 0.7, 8] for k in range(number_wire)]
scan_dict["scantype"] = scantype
scan_dict["fullcommand"] = fullcommand
scan_dict["localhdf5file"] = localhdf5file
if CCDType == 'EIGER_4MCdTe':
    scan_dict['imageOffset'] = 0 # pedestal of image intensity signal
    scan_dict["monitorOffset"] = 31.7 # MArch 2026
    scan_dict["monitorOffset"] = 28.3 # June 2026
    
else:
    scan_dict['imageOffset'] = 1000 # pedestal of image intensity signal
    scan_dict["monitorOffset"] = 25  # old data

scantype, scan_dict

In [ ]:
# example how to get data from a given scan
# import h5py

# with h5py.File(localhdf5file, 'r', locking=False) as f:
#     scannode = f[f'{scan_dict["scanNumber"]}.1']
#     data = scannode['measurement']
#     data2 = scannode['measurement/mon'][()]
#     wirep = scannode[f'instrument/positioners/zf'][()]
#     # print(data['mon'])
#     # print(data['mon'][()])
#     data2

In [ ]:
## Create the scan ## 
scan = StaticPointScan(scan_dict, verbose=False)

In [ ]:
#scan.calc_wires_range_shadow(300)

In [ ]:

print('approx. p0 value', scan.wire_position[0], scan.wire_position[-1])
print('Value to put in the next simulator GUI')
estimatedp0 = 0.5*(scan.wire_position[0]+ scan.wire_position[-1])
print(np.round(estimatedp0,2))


# SIMULATION OF WIRES SHADOWS

It can be skipped if you have already a ####_calib.scan ready for refinement

- The GUI will help to find the initial wires trajectory parameters before fitting
- a corresoponding ####_calib.scan will be produced

In [ ]:
def ShowWireShadows(imageindex,ysrc, ax):
    """ysrc  in microns"""

    xmin, xmax = 0, scan.get_img_params(['framedim'])[0]
    ys_right = scan.calc_wires_range_shadow(imageindex , xcam=xmax, ysrc=ysrc/1000.)
    ys_left = scan.calc_wires_range_shadow(imageindex, xcam=xmin, ysrc=ysrc/1000.)

    align = 'top'

    colors = mplp.rcParams['axes.prop_cycle'].by_key()['color']
    
    for i, ymin in enumerate(ys_left):
        ymax = ys_right[i]
        line = Line2D([xmin, xmax], [ymin[0], ymax[0]],
                      antialiased=True, color=colors[i], linestyle='-')
        poly = Polygon(list(zip([xmin, xmax, xmax, xmin],
                           [ymin[1], ymax[1], ymax[2], ymin[2]])),
                       closed=True, antialiased=True, alpha=0.5, facecolor=colors[i])

        txt = ax.text(xmax-10, ymin[1], str(i+1), fontsize='small',
                                   horizontalalignment='right', verticalalignment=align)

        ax.add_line(line)
        ax.add_patch(poly)

    #self._fig_canvas.draw()
    
def update_wire(index_wire,h, p0, f1, f2, R, ysrc):
    scan.set_wire(index_wire,{'material': 'W',
  'R': float(R),
  'h': float(h),
  'p0': float(p0),
  'f1': float(f1)*np.pi/180.,
  'f2': float(f2)*np.pi/180.,
  'u1': 0,
  'u2': 0.0})
    update_img(imageindex=imageindex_widget.value, showWire = True, wire_selector=wire_selector.value, ysrc=ysrc)

def reset_box(change):
    dico = scan.get_wires_dict()
    h_value.value=float(dico[int(change.new)-1]['h'])
    p0_value.value= float(dico[int(change.new)-1]['p0'])
    f1_value.value=float(dico[int(change.new)-1]['f1'])*180/np.pi
    f2_value.value=float(dico[int(change.new)-1]['f2'])*180/np.pi
    R_value.value=float(dico[int(change.new)-1]['R'])
    ysrc_value.value=float(dico[int(change.new)-1]['ysrc'])

In [ ]:
## WIDGET DEFINITION TO CALIBRATE WIRE ##
h_value = widgets.FloatText(
    value=0.7,
    description='h:',
    disabled=False
)

p0_value = widgets.FloatText(
    value=estimatedp0,
    description='p0:',
    disabled=False
)

f2_value = widgets.FloatText(
    value=0.,
    description='f2 (°):',
    disabled=False
)

f1_value = widgets.FloatText(
    value=0.,
    description='f1 (°):',
    disabled=False
)

R_value = widgets.FloatText(
    value=0.025,
    description='R:',
    disabled=False
)

ysrc_value = widgets.FloatText(
    value=0,
    description='ysource (µm)',
    disabled=False
)

vbox = widgets.VBox([h_value, p0_value, f1_value, f2_value, R_value, ysrc_value])

imageindexmax = scan.number_images-1
#imageindexmax = 800 # for simulation

imageindex_widget = widgets.IntSlider(value=0, min=0, max=imageindexmax, description="Image index")
showWire_widget = widgets.Checkbox(value=False, description="Show wire")
wire_selector = widgets.IntSlider(value=1, min=1, max=scan.wire_qty, description="Wire selector")
vmaxslider = widgets.IntSlider(value=2000, min=10, max=6000, description="vmax")
ysrcslider = widgets.IntSlider(value=0, min=-100, max=2000, description="ysrc")

ui = widgets.VBox([vmaxslider, imageindex_widget, showWire_widget, wire_selector])


In [ ]:
scan.number_images

## GUI VISUALISATION of WIRES shadows

In [ ]:
figplot, ax = mplp.subplots()
ax.set_xlim(0,scan.get_img_params(['framedim'])[0])
ax.set_ylim(scan.get_img_params(['framedim'])[1],0)


def update_img(imageindex=0, showWire = False, wire_selector=1, vmax=2000, ysrc=0):
    ymin, ymax = ax.get_ylim()
    xmin, xmax = ax.get_xlim()
    if plt.gci() is not None:
        vmin, vmax = plt.gci().get_clim()
    ax.clear()
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)

    ax.set_xlabel("Xcam")
    ax.set_ylabel("Ycam")
    if imageindex>=scan.number_images:
        imageindex = scan.number_images-1
        #print('no image available')
    paddedindex = '%s'%str(imageindex).zfill(sizeofzeropadding)
    imagefilename = f"{prefix}{paddedindex}{suffix}"

    img = fabio.open(Path(image_folder)/imagefilename).data
    c1 = int(np.mean(img, axis=(0, 1))) + 4 * int(np.std(img, axis=(0, 1)))
    #vmin, vmax = 1000, int(c1)
    vmin = 0
    
    ax.imshow(img, origin='upper', vmin=vmin, vmax=vmax, cmap='grey')
    if showWire:
        ShowWireShadows(imageindex,ysrc,ax)

    plt.show()

        
out = widgets.interactive_output(update_img, {
    "imageindex": imageindex_widget,
    "showWire": showWire_widget,
    "wire_selector": wire_selector,
    "vmax":vmaxslider,
    "ysrc":ysrcslider
})
    
wire_selector.observe(reset_box, names='value')
h_value.observe(lambda change: update_wire(wire_selector.value-1, change['new'], p0_value.value, f1_value.value, f2_value.value, R_value.value, ysrc_value.value), names='value')
p0_value.observe(lambda change: update_wire(wire_selector.value-1,h_value.value, change['new'], f1_value.value, f2_value.value, R_value.value, ysrc_value.value), names='value')
f1_value.observe(lambda change: update_wire(wire_selector.value-1,h_value.value, p0_value.value, change['new'], f2_value.value, R_value.value,ysrc_value.value), names='value')
f2_value.observe(lambda change: update_wire(wire_selector.value-1,h_value.value, p0_value.value, f1_value.value, change['new'], R_value.value,ysrc_value.value), names='value')
R_value.observe(lambda change: update_wire(wire_selector.value-1, h_value.value, p0_value.value, f1_value.value, f2_value.value, change['new'],ysrc_value.value), names='value')
ysrc_value.observe(lambda change: update_wire(wire_selector.value-1,h_value.value, p0_value.value, f1_value.value, f2_value.value, R_value.value, change['new']), names='value')
display(ui, out)
display(vbox)

## SEE the wires parameters

WARNING, f1 and f2 (tilt angles of the wires, mostly f1 is visible) are given here in radians in scan.get_wires_dict()

In [ ]:
scan.get_wires_dict()

## SET filename and SAVE a ####_calib.scan

In [ ]:
ScanID, analysis_dir, scan.scantype

In [ ]:
#ScanID: Corresponds to 'prefix' in 'header.py'
#prefix_DAXM_analysis = ScanID
prefix_DAXM_analysis = 'init'

SavedName = f'{prefix_DAXM_analysis}_calib.scan'

#=====================================================================
scan.input['wire'] = scan.get_wires_dict()

# Take too much time  !!!
point_scan = PointScan(scan.input, verbose=False)
point_scan.save(SavedName, directory=analysis_dir)
GT.printgreen(f'\n####_calib.scan file saved in {os.path.join(analysis_dir,SavedName)}')

In [ ]:
point_scan.scantype, point_scan.wire_step, point_scan.number_images, point_scan.wire_params

In [ ]:
point_scan.localhdf5file

# CALIBRATION wires trajectory parameters refinement

- use the initial parameters found above

- or restart directly from a .scan file  (either init_calib.scan or some .scan file after refinement

Following the calib.py script

In [ ]:
analysis_dir = '/data/visitor/blc17179/bm32/20260617/PROCESSED_DATA/BaTiO3/BaTiO3_daxmfluo/scan0001'
analysis_dir

In [ ]:
import os
import matplotlib.pylab as mplp
from LaueTools.Daxm.classes.scan.scan import new_scan
from LaueTools.Daxm.classes.source import new_source
from LaueTools.Daxm.classes.calibration import new_calib, CalibManager
from LaueTools.Daxm.classes.reconstruction import RecManager
print("daxm classes loaded ...")

## SET Output Folders: reconstruction and calibration

In [ ]:
from pathlib import Path

calib_dir = Path(analysis_dir+'/'+'calibration')
rec_dir = Path(analysis_dir+'/'+'reconstruction')

# Make sure directory exists
os.makedirs(calib_dir, exist_ok=True)
# Make sure directory exists
os.makedirs(rec_dir, exist_ok=True)

print(' "calib_dir" set to', calib_dir)
print(' "rec_dir" set to', rec_dir)

the following comes from script calib.py

In [ ]:
# look at already saved .scan file
!ls {analysis_dir}

In [ ]:
# edit a scan file to check
!cat {Path(analysis_dir)/'init_calib.scan'}

In [ ]:
# TODO !!!!!   read all scans !!

#SavedName ='ZrCr_ech15_aligndaxm_42_calib.scan'
#SavedName ='ZrCr_Gedaxm_6_calib.scan'
#SavedName = 'Al_mardisoir_Gedaxm_expo_0.2_nbsteps_400_hmicro1mm_1_calib.scan'
SavedName = 'init_calib.scan'

# previousscanfile_for_calib = 'Al_mardisoir_Gedaxm_expo_0.2_nbsteps_400_hmicro1mm_1_calib.scan'
# calib_file = os.path.join(analysis_dir, previousscanfile_for_calib)

calib_file = os.path.join(analysis_dir, SavedName)


#----------- load/create scan, sample, calib -----------
scan = new_scan(calib_file, verbose=False)
ScanID = scan.hdf5scanId

In [ ]:
#scan.get_wires_dict()

In [ ]:
sample = new_source("Ti", 0.5, ystep=0.001)
calib = new_calib(scan, sample, kind="fluo")

In [ ]:
# automatic selection of grid points on detector

#TODO  !!!!   repair adjust = True
# TODO  !!!!  accelerate Loading (and calculating ?) experimental intensity profiles ...

calib.set_points_grid(dims=[2, 2], adjust=False)


In [ ]:
# manual selection of points on detector
# needs to launch anyway calib.set_points_grid(dims=[2, 2], adjust=False) to init other things (nb of points ....)
# will set calib.data_XYcam

listptsW1=[[689,1280],[1308,1231],[352,1201],[1474,1365]]
listptsW2=[[689,822],[1378,822],[689,1011],[1350,950]]
listptsW3=[[689,182],[1378,182],[689,365],[1385,365]]
XYcam = [listptsW1, listptsW2, listptsW3]
calib.set_points(XYcam)

In [ ]:
wire_index = 2
calib.plot_exp_wire(wire_index)

In [ ]:
#----------- coarse calibration-----------

#bounds=([-.1,.1],[-.2,.2],[-1,1])  # degrees for axis or f1 f2 u1 u2, and SHOULD be diffeentila value around initial guess 
#calib.run(var=['h', 'p0', 'f1'])#, bounds=bounds)
# calib.run(var=['h', 'p0','axis','dm'])
# intermediatecalibfile = ScanID+"_notebook_coarse"
# calib.save_wires(intermediatecalibfile, directory=calib_dir)
# calib.log_plot()
# mplp.show()



# #-------------------------------------------
# calib.load_wires(ScanID+"_notebook_coarse_radii", directory=calib_dir, verbose=3)
# calib.log_restart()



# #calib.load_wires(intermediatecalibfile, directory=calib_dir)
# #calib.set_points_grid(dims=[2, 2], adjust=False)
# calib.run(var=['h', 'p0','R'])

# intermediatecalibfile_1 = ScanID+"_notebook_coarse_radii2"
# calib.save_wires(intermediatecalibfile_1, directory=calib_dir)
# calib.log_plot()
# mplp.show()

# ###---------------------------------------------------------
# #intermediatecalibfile_1 = 'Al_mardisoir_Gedaxm_expo_0.2_nbsteps_400_hmicro1mm_1_notebook_coarse_radii'

# #----------- coarse calibration-----------
# calib.load_wires(intermediatecalibfile_1,directory=calib_dir, verbose=3)
# calib.log_restart()
# calib.set_points_grid(dims=[3, 3], adjust=False)
# calib.run(var=['h', 'p0', 'axis', 'R', 'u2', 'dm'], verbose=3)
# intermediatecalibfile2a = ScanID+"_notebook_lesscoarse"
# calib.save_wires(intermediatecalibfile2a, directory=calib_dir)
# calib.log_plot()
# mplp.show()

#----------- coarse calibration-----------
calib.load_wires(intermediatecalibfile2a,directory=calib_dir, verbose=3)
calib.log_restart()
calib.set_points_grid(dims=[3, 3], adjust=False)
calib.run(var=['h', 'p0', 'axis', 'R', 'u1', 'u2', 'dm'], verbose=3)
intermediatecalibfile2b = ScanID+"_notebook_lesscoarse"
calib.save_wires(intermediatecalibfile2b, directory=calib_dir)
calib.log_plot()
mplp.show()

#----------- fine calibration-----------

calib.load_wires(intermediatecalibfile2b, directory=calib_dir)
calib.log_restart()
calib.data_span = 3.5
calib.set_points_grid(dims=[6, 6],adjust=False)
calib.run(var=['R', 'h', 'p0', 'axis', 'u1', 'u2', 'dm'])
finalcalibfile = ScanID+"_notebook"
calib.save_wires(finalcalibfile, directory=calib_dir)
calib.log_plot()
mplp.show()


In [ ]:
# calib.log_plot()
# mplp.show()

In [ ]:
#Al_mardisoir_Gedaxm_expo_0.2_nbsteps_400_hmicro1mm_1_calib

In [ ]:
calib.load_wires(ScanID+"_notebook", directory=calib_dir)
calib.log_restart()
calib.data_span = 3.5
calib.set_points_grid(dims=[6, 6],adjust=False)
calib.run(var=['R', 'h', 'p0', 'axis', 'u1', 'u2', 'dm'])
finalcalibfile = ScanID+"_notebook_allparams"
calib.save_wires(finalcalibfile, directory=calib_dir)
calib.log_plot()
mplp.show()

In [ ]:
calib.data_XYcam

# Test performance

In [ ]:
import cProfile
import pstats
import numpy as np

def profile_code():
    # Your code here
    XYcam = np.concatenate(calib.data_XYcam)
    wire = np.concatenate([[i] * len(xy) for i, xy in enumerate(calib.data_XYcam)])
    data_I, data_pw = calib.scan.get_profile_manypixels_centred(wire, XYcam, calib.data_hbs, calib.data_span)

    for i, wid in enumerate(wire):
        calib.data_pw[wid].append(data_pw[i])
        calib.data_I[wid].append(data_I[i])

# Run the profiler
cProfile.runctx('profile_code()', globals(), locals(), 'profile_stats')

# Print the profiling results
stats = pstats.Stats('profile_stats')
stats.strip_dirs().sort_stats('cumtime').print_stats(10)  # Show top 10 time-consuming functions